In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import os

# 1. Define paths
processed_data_path = '../data/processed/ethiopia_fi_enriched.csv'
weights_path = '../data/processed/calibrated_impact_weights.csv'

# 2. Load the data
try:
    # Load historical data
    df_hist = pd.read_csv(processed_data_path)
    df_hist['observation_date'] = pd.to_datetime(df_hist['observation_date'], format='mixed')
    
    # Load the "Intelligence" from Task 3
    # index_col=0 ensures the Event names (like 'Telebirr Launch') are the index
    weights = pd.read_csv(weights_path, index_col=0)
    
    print(f"✅ Success: Data and Weights loaded.")
    print(f"Historical Records: {len(df_hist)}")
    print(f"Calibrated Events: {len(weights)}")
    
    # Show the events we have weights for
    print("\nAvailable Events for Forecasting:")
    print(weights.index.tolist())

except Exception as e:
    print(f"❌ Error: {e}")

# Display first few weights to verify calibration was applied
display(weights.head(3))

In [ ]:
# --- Cell 2: Baseline and Future Event Scheduling ---

# 1. Identify Starting Values (The 2024 Baseline)
# We look for the most recent values for our two core dimensions
latest_access = df_hist[df_hist['indicator_code'] == 'ACC_OWNERSHIP']['value_numeric'].max()

# For Usage, we look at Digital Payments or use the 35% baseline if data is sparse
usage_indicator = 'USG_P2P_COUNT' # Using P2P counts as a proxy for Usage growth
latest_usage = 35.0 # Starting point for USG_DIGITAL_PAYMENT based on Global Findex 2024

print(f"📊 2024 Baseline: Access = {latest_access}%, Usage = {latest_usage}%")

# 2. Schedule Future Events (2025 - 2027)
# Based on the National Financial Inclusion Strategy (NFIS-II) timeline
future_schedule = {
    2025: ['Fayda Digital ID Program Rollout', 'Safaricom Ethiopia Price Increase'],
    2026: ['EthioPay Instant Payment System Launch', 'Foreign Exchange Liberalization'],
    2027: ['M-Pesa EthSwitch Integration']
}

# 3. Define Natural Growth (Background trend without major events)
# Ethiopia's natural inclusion growth (historical trend) is roughly 1.0% per year
natural_growth_rate = 1.0 

print("\n✅ Baseline set and future events scheduled.")

In [ ]:
# --- Cell 3: The Forecasting Engine ---

# 1. Initialize results with the 2024 baseline
forecast_results = [
    {'Year': 2024, 'Access (%)': latest_access, 'Usage (%)': latest_usage, 'Type': 'Actual'}
]

# 2. Tracking variables
current_access = latest_access
current_usage = latest_usage

# 3. Run the Forecast Loop
for year in [2025, 2026, 2027]:
    # A. Natural Trend (Background growth)
    # Access grows naturally at 1%, Usage (digital) grows faster at 2%
    access_lift = 1.0 
    usage_lift = 2.0  
    
    # B. Add Lifts from Scheduled Events
    year_events = future_schedule.get(year, [])
    
    for event_name in year_events:
        if event_name in weights.index:
            # Add weight for Access (Account Ownership)
            access_lift += weights.loc[event_name, 'ACC_OWNERSHIP'] if 'ACC_OWNERSHIP' in weights.columns else 0
            
            # Add weight for Usage (Taking the maximum value among usage indicators)
            # This looks at USG_P2P_COUNT, USG_TELEBIRR_USERS, etc.
            usage_cols = [c for c in weights.columns if c.startswith('USG_')]
            if usage_cols:
                usage_lift += weights.loc[event_name, usage_cols].max()
    
    # Update current totals
    current_access += access_lift
    current_usage += usage_lift
    
    # Store results
    forecast_results.append({
        'Year': year, 
        'Access (%)': round(current_access, 2), 
        'Usage (%)': round(current_usage, 2), 
        'Type': 'Forecast'
    })

# 4. Convert to DataFrame
df_forecast = pd.DataFrame(forecast_results)

print("📈 Forecast Generated for 2025-2027:")
display(df_forecast)

# Save for the Dashboard task
os.makedirs('../data/processed', exist_ok=True)
df_forecast.to_csv('../data/processed/ethiopia_fi_forecast_2027.csv', index=False)

In [2]:
# --- Cell 4: Robust Forecasting Visualization ---
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import os

# 1. Load the forecast data from the CSV we saved in Step 3
forecast_csv_path = '../data/processed/ethiopia_fi_forecast_2027.csv'

if os.path.exists(forecast_csv_path):
    df_plot = pd.read_csv(forecast_csv_path)
    print("✅ Forecast data loaded from CSV.")
else:
    print("❌ Error: Forecast CSV not found. Please re-run Cell 3.")

# 2. Create the figure
fig = go.Figure()

# 3. Add Access Forecast
fig.add_trace(go.Scatter(
    x=df_plot['Year'], 
    y=df_plot['Access (%)'],
    mode='lines+markers',
    name='Access (Account Ownership)',
    line=dict(color='royalblue', width=4),
    marker=dict(size=10)
))

# 4. Add Usage Forecast
fig.add_trace(go.Scatter(
    x=df_plot['Year'], 
    y=df_plot['Usage (%)'],
    mode='lines+markers',
    name='Usage (Digital Payments)',
    line=dict(color='firebrick', width=4, dash='dash'),
    marker=dict(size=10)
))

# 5. Add the 60% Policy Target Line
fig.add_hline(
    y=60, 
    line_dash="dot",
    annotation_text="NFIS-II Target (60%)", 
    annotation_position="bottom right",
    line_color="green",
    line_width=2
)

# 6. Add Scenario Shading
fig.add_vrect(
    x0=2024.5, x1=2027.5,
    fillcolor="gray", opacity=0.1,
    layer="below", line_width=0,
    annotation_text="FORECAST PERIOD", annotation_position="top left"
)

# 7. Styling
fig.update_layout(
    title='Ethiopia Financial Inclusion Forecast: 2025 - 2027',
    xaxis=dict(tickmode='linear', tick0=2024, dtick=1),
    yaxis_title='Percentage of Adult Population (%)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# Show the interactive chart
fig.show()

# (Optional) Try to save, but catch error if Kaleido is missing
try:
    fig.write_image("../reports/figures/inclusion_forecast_2027.png")
    print("✅ Image saved to reports/figures/")
except Exception:
    print("⚠️ Note: Image not saved (Kaleido missing), but chart is displayed above.")

✅ Forecast data loaded from CSV.


✅ Image saved to reports/figures/
